## Ingesting Documents for Cloud ChromaDB

In [1]:
from langchain_community.document_loaders.csv_loader import CSVLoader

hyperbook_docs = CSVLoader(
    file_path='../../data/raw/hyperbook.csv', 
    content_columns=['content'], 
    metadata_columns=['url', 'title', 'authors'],
    encoding='utf-8'
).load()

In [2]:
from pathlib import Path
from langchain_community.document_loaders.pdf import PyPDFLoader

textbooks_docs = []

ball_metadata = {
    'url': 'https://mcgill.on.worldcat.org/oclc/180470082',
    'title': "Shapes: Nature's patterns: a tapestry in three parts",
    'authors': 'Philip Ball'
}

def set_metadata(doc):
    doc.metadata = ball_metadata
    return doc

# Temporary — Only experimenting with three chapters from Shapes by Ball, P.
for chapter_path in Path('../../data/raw/textbooks/shapes_ball').iterdir():
    chapter_docs = PyPDFLoader(chapter_path).load()
    chapter_docs = list(map(set_metadata, chapter_docs))
    textbooks_docs = textbooks_docs + chapter_docs 

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = hyperbook_docs + textbooks_docs

# Chunk parameters
chunk_size = 256
chunk_overlap = 32

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
doc_splits = text_splitter.split_documents(docs)

In [4]:
import pandas as pd

df = pd.DataFrame({
    'content': [doc.page_content for doc in doc_splits],
    'url': [doc.metadata['url'] for doc in doc_splits],
    'title': [doc.metadata['title'] for doc in doc_splits],
    'authors': [doc.metadata['authors'] for doc in doc_splits]
})

In [5]:
df.to_csv('../../data/processed/bien200_chunks.csv', index=False)

In [14]:
import os
import chromadb
from langchain_chroma.vectorstores import Chroma
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=1024
)

chroma_client = chromadb.CloudClient(
    tenant=os.getenv('CHROMA_TENANT'),
    database=os.getenv('CHROMA_DATABASE'),
    api_key=os.getenv('CHROMA_API_KEY')
)

vector_store = Chroma(
    collection_name='bien200', 
    embedding_function=embeddings,
    client=chroma_client
)

In [ ]:
from uuid import uuid4

def split_list(input_list, chunk_size=300):
    for i in range(0, len(input_list), chunk_size):
        yield input_list[i:i + chunk_size]

for batch_docs in split_list(doc_splits):
    batch_ids = [str(uuid4()) for _ in range(len(doc_splits))]
    vector_store.add_documents(documents=batch_docs, ids=batch_ids)

In [11]:
dense_retriever = vector_store.as_retriever()

In [12]:
dense_retriever.invoke('signs for recognizing extraterrestrial life')

[Document(id='fca2d8f8-8509-4920-a6d5-ae7ae0d61aa7', metadata={'authors': 'Philip Ball', 'title': "Shapes: Nature's patterns: a tapestry in three parts", 'url': 'https://mcgill.on.worldcat.org/oclc/180470082'}, page_content='The Shapes of Things\nPattern and Form\nARRIVING on Earth, the aliens approach the first thing they see and utter the familiar words: ‘take \nme to your leader’ ( ). Like many jokes, this one offers a damning critique. It undermines the Fig. 1.1\nvenerable and serious scientific quest to find life on other worlds, exploding the question of ‘how \nwould we  if we found it?’ by answering that we tend to imagine it will look like us.know\nNow, let me assure you that astrobiologists (as scientists who study aliens are called nowadays) \nare not really that foolish. They do not imagine for a moment that when we touch down on another \ninhabited world, we will be greeted by envoys who look like Leonard Nimoy. Indeed, if there is life \nin those parts of our own solar sys

## Ingesting Documents for Local ChromaDB

In [4]:
import chromadb

chroma_client = chromadb.HttpClient(host='localhost', port=8000)
chroma_client.heartbeat()

1755290780162321566

In [ ]:
chroma_client.list_collections()

[Collection(name=bien200)]

: 

In [7]:
collection = chroma_client.get_collection(name="bien200")

In [8]:
import os
import chromadb
from langchain_chroma.vectorstores import Chroma
from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=1024
)

In [9]:
vector_store = Chroma(
    collection_name='bien200', 
    embedding_function=embeddings,
    client=chroma_client
)

In [ ]:
from uuid import uuid4

def split_list(input_list, chunk_size=300):
    for i in range(0, len(input_list), chunk_size):
        yield input_list[i:i + chunk_size]

for batch_docs in split_list(doc_splits):
    batch_ids = [str(uuid4()) for _ in range(len(doc_splits))]
    vector_store.add_documents(documents=batch_docs, ids=batch_ids)

In [10]:
dense_retriever = vector_store.as_retriever()

In [ ]:
dense_retriever.invoke('the essence of humanity')

[Document(id='13b35d65-8c04-4d5c-afc4-acf35c23f357', metadata={'authors': 'Philip Ball', 'title': "Shapes: Nature's patterns: a tapestry in three parts", 'url': 'https://mcgill.on.worldcat.org/oclc/180470082'}, page_content='of history as spiritual destiny, Haeckel implied that there was an organizing force that shaped nature \nall the way from protoplasm (primordial living matter) to humankind. Indeed, he felt that even our \nmental and spiritual states are direct products of evolution: somewhere along this great chain of \nbeing we acquired a soul. Our appreciation of beauty and our capacity for mathematical abstraction \nboth emerged from this tendency of nature to organize itself into ever more complex forms, based on \nsymmetry and pattern. Thus Haeckel held a somewhat Platonic view of beauty, which is not in the \neye of the beholder but is the result of the way we are conditioned by the objective forms of the \nnatural world. In his bestselling  (1899),Haeckel asserted that ther

In [3]:
import os

os.getenv('RANDOM')

In [1]:
import os

os.getenv('CHROMA_LOCAL')

'true'